# 06 · Correlation, VIF & Encoding — FRAUD_ECOMMERCE
**Marker:** `fraud_nb01-07_v2` · **Input:** `data/05_hypothesis_done.parquet`, `reports/05_run_record.json`
**Outputs:** `data/06_encoded_tree.parquet`, `data/06_encoded_linear.parquet`, `data/06_feature_names.parquet`, `data/06_encoding_params.json`, `reports/06_run_record.json`

* **Fail-closed on notebook 05.** Unacknowledged leakage sentinels stop this notebook. So do velocity columns that are not point-in-time.
* **The feature set is fixed by name.** Every exclusion carries a reason, and each structural reason is asserted on train rows. Correlation and VIF are printed for information only and never select features.
* **Every fitted statistic uses `__split == 'train'` only.** This covers category levels, medians, means and standard deviations. The fit function takes the whole frame and filters internally, and 06.6 proves that test rows cannot move it.
* **Two encodings:**
  * **Tree:** raw numerics with structural NaN kept, and pandas `category` with train levels.
  * **Linear:** log1p/cyclic transforms, train-median imputation with missing indicators, standardisation, and one-hot on train levels.

In [ ]:
%pip install -q boto3==1.43.95

In [ ]:
# MARKER: fraud_nb01-07_v2 :: 06_Correlation_VIF_Encoding
import io, os, json, time, hashlib, platform, importlib
from datetime import datetime, timezone
import boto3
from botocore.exceptions import ClientError
import joblib
import numpy as np
import pandas as pd
from google.colab import userdata

BUCKET, REGION = "fraud-ecommerce", "ap-south-2"
SPLIT_DATE = pd.Timestamp("2025-07-01")      # stamped in notebook 01 as __split; verified here, never recomputed
CONTRACT_VERSION = "v1"
SEED = 42
MARKER = "fraud_nb01-07_v2"
for _k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    if not os.environ.get(_k):
        os.environ[_k] = userdata.get(_k)     # Colab Secrets -> process env only; never printed or saved
s3 = boto3.client("s3", region_name=REGION)
RAW, LABELS, CONTRACTS, DATA, REPORTS = "raw/", "raw/label_sources/", "contracts/", "data/", "reports/"  # flat layout
ident = boto3.client("sts", region_name=REGION).get_caller_identity()
print("account:", ident["Account"], "| arn:", ident["Arn"])
if ident["Arn"].endswith(":root"):
    print("NOTE: running as root — accepted for Phase 1; move to an IAM principal before Phase 2")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)


def _jsonable(o):
    if isinstance(o, (np.integer, np.floating, np.bool_)):
        return o.item()
    if isinstance(o, (pd.Timestamp, datetime)):
        return o.isoformat()
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"not JSON-serialisable: {type(o).__name__}")


def read_bytes_s3(key):
    return s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()


def put_bytes_s3(body, key):
    s3.put_object(Bucket=BUCKET, Key=key, Body=body)
    print(f"saved s3://{BUCKET}/{key}  ({len(body):,} bytes)")


def key_exists(key):
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] in ("404", "NoSuchKey", "NotFound"):
            return False
        raise


def read_s3(key):
    return pd.read_parquet(io.BytesIO(read_bytes_s3(key)))


def save_s3(df, key):
    """Parquet only; the bytes are verified to round-trip columns, dtypes and categories before upload."""
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    body = buf.getvalue()
    back = pd.read_parquet(io.BytesIO(body))
    assert list(back.columns) == list(df.columns) and len(back) == len(df), f"parquet round-trip changed shape: {key}"
    bad = [c for c in df.columns if str(back[c].dtype) != str(df[c].dtype)]
    assert not bad, f"parquet round-trip changed dtypes in {key}: {bad}"
    badcat = [c for c in df.columns if str(df[c].dtype) == "category"
              and list(back[c].cat.categories) != list(df[c].cat.categories)]
    assert not badcat, f"parquet round-trip changed categories in {key}: {badcat}"
    put_bytes_s3(body, key)


def read_json_s3(key):
    return json.loads(read_bytes_s3(key))


def save_json_s3(obj, key):
    put_bytes_s3(json.dumps(obj, indent=1, default=_jsonable).encode(), key)


def save_model_s3(obj, key):
    buf = io.BytesIO()
    joblib.dump(obj, buf)
    put_bytes_s3(buf.getvalue(), key)


def load_model_s3(key):
    return joblib.load(io.BytesIO(read_bytes_s3(key)))


# names used by notebooks 01-04 (same verified implementations underneath)
def s3_read_csv(key, **kw):
    return pd.read_csv(io.BytesIO(read_bytes_s3(key)), **kw)


s3_read_parquet, s3_read_json = read_s3, read_json_s3


def s3_write_parquet(df, key):
    save_s3(df, key)
    return f"s3://{BUCKET}/{key}"


def s3_write_json(obj, key):
    save_json_s3(obj, key)
    return f"s3://{BUCKET}/{key}"


RAW_KEYS = ([f"{RAW}{t}.csv" for t in ["payments", "orders", "order_items", "account_logins", "customers",
                                         "merchants", "cards", "devices", "ip_reputation"]]
            + [f"{LABELS}{t}.csv" for t in ["fraud_events", "audit_sample", "chargebacks"]]
            + [f"{CONTRACTS}schema_v1.json"])
_absent = [k for k in RAW_KEYS if not key_exists(k)]
assert not _absent, f"missing landing objects in s3://{BUCKET}/: {_absent}"
print(f"landing objects present: {len(RAW_KEYS)} (flat layout, bucket root)")


def run_meta(notebook):
    return {"notebook": notebook, "marker": MARKER,
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "library_versions": LIB_VERSIONS, "version_drift": VERSION_DRIFT}


LIBS = ["pandas", "numpy", "pyarrow", "scipy", "statsmodels", "sklearn", "lightgbm", "xgboost", "joblib", "boto3"]
LIB_VERSIONS = {"python": platform.python_version(),
                **{m: importlib.import_module(m).__version__ for m in LIBS}}
EXPECTED = {"pandas": "2.2.3", "numpy": "2.1.3", "pyarrow": "23.0.1", "scipy": "1.16.3", "statsmodels": "0.15.0",
            "sklearn": "1.6.1", "lightgbm": "4.6.0", "xgboost": "3.4.1", "boto3": "1.43.95"}
VERSION_DRIFT = {m: {"verified": v, "found": LIB_VERSIONS[m]} for m, v in EXPECTED.items() if LIB_VERSIONS[m] != v}
if not LIB_VERSIONS["python"].startswith("3.13."):
    VERSION_DRIFT["python"] = {"verified": "3.13.x", "found": LIB_VERSIONS["python"]}
print(LIB_VERSIONS)
print("VERSION DRIFT vs the runtime verified on 2026-09-16:", VERSION_DRIFT or "none")

In [ ]:
# 06.1 Load, gate on notebook 05, re-verify velocity columns
df = read_s3("data/05_hypothesis_done.parquet")
rec05 = read_json_s3("reports/05_run_record.json")
assert rec05["marker"] == MARKER, f"05 run record is from generation {rec05.get('marker')}, expected {MARKER}"
assert df.shape == (rec05["rows"], rec05["cols"]), (df.shape, rec05["rows"], rec05["cols"])

# Fill ONLY after reviewing a flagged sentinel, e.g. {"S2_zero_fee_non_cod": "reviewed by NAME on DATE: reason"}
ACKNOWLEDGED_SENTINELS = {}
flags = {k for k, v in rec05["sentinels"].items() if v["flag"]}
assert "S3_velocity_not_point_in_time" not in flags, \
    "velocity columns are full-frame aggregates: re-run 02 -> 03 -> 04 -> 05 from package fraud_nb01-07_v2"
unack = sorted(flags - set(ACKNOWLEDGED_SENTINELS))
assert not unack, f"unreviewed leakage sentinels from notebook 05: {unack}"
print("05 sentinels:", {k: v["flag"] for k, v in rec05["sentinels"].items()}, "| acknowledged:", ACKNOWLEDGED_SENTINELS)

def asof_distinct_count(frame, key, entity="customer_id", ts="payment_ts"):
    """Distinct `entity` values seen on `key` at or before each row's `ts` (inclusive). NaN where `key` is null.
    Point-in-time by construction: a row never sees a later first appearance, so no split boundary is needed."""
    f = frame.loc[frame[key].notna(), [key, entity, ts]]
    first = (f.groupby([key, entity], as_index=False, sort=True)[ts].min()
               .sort_values(ts, kind="mergesort"))
    first["_n"] = first.groupby(key, sort=False).cumcount() + 1
    rows = f.assign(_row=f.index).sort_values(ts, kind="mergesort")
    m = pd.merge_asof(rows, first[[key, ts, "_n"]], on=ts, by=key,
                      direction="backward", allow_exact_matches=True)
    assert m["_n"].notna().all(), f"as-of join left gaps for {key}"
    return m.set_index("_row")["_n"].reindex(frame.index).astype("float64")


VELOCITY = {"device_n_customers": "device_id", "ip_n_customers": "ip_address"}


for col, key in VELOCITY.items():
    stored = df[col].astype("float64")
    pit = asof_distinct_count(df, key)
    assert stored.fillna(-1).eq(pit.fillna(-1)).all(), f"{col} is not the as-of distinct count"
print("velocity columns verified point-in-time")

In [ ]:
# 06.2 Baseline columns and dtype normalisation (row-preserving, target-free)
df["baseline_score"] = pd.to_numeric(df["payment_risk_score"].str.strip(), errors="raise").astype("float64")
df["alerted_int"] = df["alerted_flag"].map({"Y": 1, "N": 0}).astype("int64")
for c in ["is_guest_checkout", "is_3ds_attempted", "is_3ds_success"]:     # pandas nullable Float64 -> float64
    assert df[c].notna().all(), c
    df[c] = df[c].astype("float64")
tr = df["__split"].eq("train")
T = df.loc[tr]
print(df.shape, "| train", int(tr.sum()))

In [ ]:
# 06.3 Structural identities behind the by-name exclusions — asserted on train rows
IDENTITIES = []


def must_equal(name, a, b):
    a = pd.Series(np.asarray(a, dtype=object)); b = pd.Series(np.asarray(b, dtype=object))
    bad = int((~((a == b) | (a.isna() & b.isna()))).sum())
    assert bad == 0, f"identity '{name}' fails on {bad} train rows — the exclusion list is no longer valid"
    IDENTITIES.append({"identity": name, "mismatches": 0})


def must_be_function(name, key, val):
    g = T.dropna(subset=[key]).groupby(key)[val].nunique(dropna=False)
    assert (g <= 1).all(), f"'{name}' is not a function: {g[g > 1].to_dict()}"
    IDENTITIES.append({"identity": name, "mismatches": 0})


def must_nearly_equal(name, mismatch_mask, max_share):
    share = float(np.mean(mismatch_mask))
    assert share <= max_share, f"'{name}': {share:.4%} of train rows differ (allowed {max_share:.2%})"
    IDENTITIES.append({"identity": name, "mismatches": int(np.sum(mismatch_mask)), "share": share})


ic = T["ip_country"]
must_equal("primary_category == merchant_category", T.primary_category, T.merchant_category)
must_be_function("mcc -> merchant_category", "mcc", "merchant_category")
must_be_function("merchant_category -> mcc", "merchant_category", "mcc")
must_be_function("shipping_speed -> shipping_charge", "shipping_speed", "shipping_charge")
must_be_function("device_type -> os_family", "device_type", "os_family")
must_equal("os_family isna == device_type isna", T.os_family.isna(), T.device_type.isna())
must_equal("bill_ship_mismatch == 1 - address_match_flag", T.bill_ship_mismatch, 1 - T.address_match_flag)
must_equal("device_orphan == 1 - has_device_profile", T.device_orphan, 1 - T.has_device_profile)
must_equal("is_disposable_email == email_domain_class=='disposable'", T.is_disposable_email,
           T.email_domain_class.eq("disposable").astype(int))
must_equal("is_hosting == asn_type in {Hosting,VPN}", T.is_hosting, T.asn_type.isin(["Hosting", "VPN"]).astype(float))
must_equal("basket_value isna == has_basket==0", T.basket_value.isna(), T.has_basket.eq(0))
must_nearly_equal("basket_value == order_value (basket rows)",
                  (T.basket_value - T.order_value).abs().gt(1e-6) & T.basket_value.notna(), 0.0)
must_equal("ip_country_missing == ip_country=='Unknown'", T.ip_country_missing, ic.eq("Unknown").astype(int))
must_equal("ip_country_mismatch == ip_country not in {IN,Unknown}", T.ip_country_mismatch,
           (~ic.isin(["IN", "Unknown"])).astype(float))
must_equal("issuer_foreign == issuing_country != 'IN' (NaN without card)", T.issuer_foreign,
           np.where(T.issuing_country.isna(), np.nan, T.issuing_country.ne("IN").astype(float)))
must_equal("payment_gateway=='Unknown' == payment_method=='Cash on Delivery'",
           T.payment_gateway.eq("Unknown"), T.payment_method.eq("Cash on Delivery"))
must_equal("has_card == card_token notna", T.has_card, T.card_token.notna().astype(int))
must_equal("has_coupon == coupon_code notna", T.has_coupon, T.coupon_code.notna().astype(int))
must_equal("hour_of_day == payment_ts.hour", T.hour_of_day, T.payment_ts.dt.hour)
must_equal("day_of_week == payment_ts.dayofweek", T.day_of_week, T.payment_ts.dt.dayofweek)
must_equal("is_weekend == day_of_week >= 5", T.is_weekend, T.day_of_week.ge(5).astype(int))
must_equal("is_night == hour_of_day in 0..5", T.is_night, T.hour_of_day.between(0, 5).astype(int))
must_nearly_equal("payment_amount == order_value - discount + shipping_charge",
                  (T.payment_amount - (T.order_value - T.discount_amount + T.shipping_charge)).abs().gt(0.01), 0.001)
must_nearly_equal("item_count == n_lines", T.item_count.astype(float).ne(T.n_lines) & T.n_lines.notna(), 0.002)
_card = T.has_card.eq(1)
must_nearly_equal("is_prepaid == product_type=='Prepaid' (card rows)",
                  T.is_prepaid[_card].ne(T.product_type[_card].eq("Prepaid").astype(float)), 0.002)
print(pd.DataFrame(IDENTITIES).to_string(index=False))

In [ ]:
TREE_NUMERIC = [
    "payment_amount", "processing_fee", "discount_amount", "item_count", "attempt_seq_in_session",
    "is_guest_checkout", "is_3ds_attempted", "is_3ds_success",
    "address_match_flag", "shipping_addr_age_hours", "kyc_level", "city_tier",
    "prior_return_rate", "prior_orders_12m", "avg_ticket_size", "trailing_chargeback_rate_bps",
    "is_emulator", "reputation_score", "n_categories", "max_unit_price", "total_qty",
    "last_login_new_device", "last_login_unusual_loc", "last_login_failed_attempts", "last_login_risk",
    "hours_since_last_login", "card_token_age_h", "device_age_h", "account_age_days",
    "amount_vs_merchant_ticket", "ip_country_mismatch", "ip_country_missing", "ip_region_mismatch",
    "issuer_foreign", "has_coupon", "hour_of_day", "day_of_week",
    "device_n_customers", "ip_n_customers",
    "has_card", "has_device_profile", "has_basket", "has_prior_login",
]
TREE_CATEGORICAL = [
    "payment_method", "payment_gateway", "shipping_speed", "delivery_type", "email_domain_class",
    "acquisition_channel", "merchant_category", "network", "issuer", "product_type",
    "device_type", "browser_family", "asn_type", "asn_country",
]


# Linear family: same information, transformed for a linear model
LINEAR_NUMERIC = [
    "log_payment_amount", "log_max_unit_price", "log_amount_vs_merchant_ticket",
    "item_count", "attempt_seq_in_session", "is_guest_checkout", "is_3ds_attempted", "is_3ds_success",
    "address_match_flag", "shipping_addr_age_hours", "kyc_level", "city_tier", "prior_return_rate",
    "trailing_chargeback_rate_bps", "is_emulator", "reputation_score", "n_categories", "total_qty",
    "last_login_new_device", "last_login_unusual_loc", "last_login_failed_attempts", "last_login_risk",
    "account_age_days", "ip_country_mismatch", "ip_country_missing", "ip_region_mismatch", "issuer_foreign",
    "has_coupon", "is_weekend", "is_night", "has_card", "has_device_profile", "has_basket", "has_prior_login",
]
LINEAR_LOG1P = ["processing_fee", "discount_amount", "prior_orders_12m", "avg_ticket_size", "hours_since_last_login",
                "card_token_age_h", "device_age_h", "device_n_customers", "ip_n_customers"]
LINEAR_CYCLIC = {"hour_of_day": 24}                     # -> hour_of_day_sin, hour_of_day_cos
LINEAR_CATEGORICAL = TREE_CATEGORICAL
LINEAR_DROP_LEVELS = {"payment_gateway": ["Unknown"]}   # == payment_method 'Cash on Delivery' (asserted in 06.3)
# structural NaN -> an existing indicator already encodes it (asserted below); other NaN -> explicit indicator
COVERED_BY_INDICATOR = {
    "log1p_card_token_age_h": "has_card", "issuer_foreign": "has_card",
    "log1p_device_age_h": "has_device_profile", "is_emulator": "has_device_profile",
    "log1p_hours_since_last_login": "has_prior_login", "last_login_new_device": "has_prior_login",
    "last_login_unusual_loc": "has_prior_login",
    "n_categories": "has_basket", "total_qty": "has_basket", "log_max_unit_price": "has_basket",
}
EXPLICIT_INDICATORS = ["last_login_failed_attempts", "last_login_risk", "log1p_device_n_customers"]

TREE_NAN_OK = ["card_token_age_h", "issuer_foreign", "device_age_h", "is_emulator", "hours_since_last_login",
               "last_login_new_device", "last_login_unusual_loc", "n_categories", "total_qty", "max_unit_price",
               "last_login_failed_attempts", "last_login_risk", "device_n_customers",
               "network", "issuer", "product_type", "device_type", "browser_family", "asn_country"]

META = ["payment_id", "payment_ts", "__split", "is_fraud", "sample_weight", "label_source", "city",
        "baseline_score", "alerted_int"]

EXCLUDED = {
    **{c: "identifier / high-cardinality key" for c in [
        "order_id", "customer_id", "merchant_id", "card_token", "device_id", "ip_address", "session_id",
        "ip_asn", "coupon_code", "bin"]},
    **{c: "high-cardinality geographic identifier; relation kept via bill_ship_mismatch/address_match_flag; DPDP proxy"
       for c in ["shipping_pincode", "billing_pincode", "home_pincode"]},
    **{c: "raw timestamp; ages / hour / weekday already derived" for c in [
        "signup_timestamp", "token_first_seen_timestamp", "first_seen_timestamp"]},
    "onboarded_date": "unparsed object dates; merchant age not derived (Phase-3 candidate)",
    "is_retry": "label-join artefact (locked decision)",
    "payment_risk_score": "incumbent output — baseline only (carried as baseline_score)",
    "alerted_flag": "incumbent output — baseline only (carried as alerted_int)",
    "primary_category": "collinear: == merchant_category (asserted)",
    "mcc": "collinear: 1:1 with merchant_category (asserted)",
    "shipping_charge": "collinear: function of shipping_speed (asserted)",
    "os_family": "collinear: function of device_type (asserted)",
    "bill_ship_mismatch": "collinear: 1 - address_match_flag (asserted)",
    "device_orphan": "collinear: 1 - has_device_profile (asserted)",
    "is_disposable_email": "collinear: email_domain_class == 'disposable' (asserted)",
    "is_hosting": "collinear: asn_type in {Hosting, VPN} (asserted)",
    "basket_value": "collinear: == order_value on basket rows (asserted)",
    "basket_value_w995": "variant of basket_value", "log_basket_value": "variant of basket_value",
    "ip_country": "reduced to ip_country_mismatch + ip_country_missing (asserted); foreign levels are tiny",
    "issuing_country": "reduced to issuer_foreign (asserted); foreign levels are tiny",
    "order_value": "near-identity: payment_amount + discount - shipping_charge (asserted <= 0.1% rows off)",
    "n_lines": "near-duplicate of item_count (asserted <= 0.2% rows off)",
    "is_prepaid": "near-duplicate of product_type == 'Prepaid' (asserted <= 0.2% card rows off)",
    "ip_region_code": "nominal region code, uniform levels; relation kept via ip_region_mismatch",
    "home_region_code": "nominal region code, uniform levels; relation kept via ip_region_mismatch",
    "payment_amount_w995": "unused variant (tree: raw; linear: log1p)",
    "max_unit_price_w995": "unused variant (tree: raw; linear: log1p)",
    "amount_vs_merchant_ticket_w995": "unused variant (tree: raw; linear: log1p)",
}

LINEAR_SOURCES = set(LINEAR_NUMERIC) | set(LINEAR_LOG1P) | set(LINEAR_CYCLIC) | set(LINEAR_CATEGORICAL)
used = set(TREE_NUMERIC) | set(TREE_CATEGORICAL) | LINEAR_SOURCES
source_meta = {"payment_id", "payment_ts", "__split", "is_fraud", "sample_weight", "label_source", "city"}
overlap = (used & set(EXCLUDED)) | (used & source_meta) | (set(EXCLUDED) & source_meta)
assert not overlap, f"columns assigned twice: {sorted(overlap)}"
unassigned = set(df.columns) - used - set(EXCLUDED) - source_meta - {"baseline_score", "alerted_int"}
assert not unassigned, f"columns with no decision: {sorted(unassigned)}"
phantom = (used | set(EXCLUDED) | source_meta) - set(df.columns)
assert not phantom, f"decisions for columns that do not exist: {sorted(phantom)}"
assert "__split" not in used
print(f"tree: {len(TREE_NUMERIC)} numeric + {len(TREE_CATEGORICAL)} categorical | "
      f"linear sources: {len(LINEAR_SOURCES)} | excluded: {len(EXCLUDED)} | meta: {len(META)}")

In [ ]:
# 06.4 Correlation among tree numerics (Spearman, deterministic 100k train sample) — information only
S = T[TREE_NUMERIC].astype("float64").sample(n=min(100_000, len(T)), random_state=SEED)
rho = S.corr(method="spearman")
pairs = (rho.where(np.triu(np.ones(rho.shape, dtype=bool), k=1)).stack()
            .rename("rho").reset_index().rename(columns={"level_0": "a", "level_1": "b"}))
high_corr = pairs.loc[pairs.rho.abs() >= 0.9].sort_values("rho", key=np.abs, ascending=False)
print("pairs with |rho| >= 0.9 (kept deliberately where listed; trees are indifferent, linear uses L2):")
print(high_corr.round(4).to_string(index=False) if len(high_corr) else "  none")

In [ ]:
# 06.5 Encoders — every statistic is fitted on __split == 'train' inside fit_encoding_params
def _levels(s):
    return sorted(s.dropna().astype(str).unique().tolist())


def linear_numeric_frame(frame):
    cols = {}
    for c in LINEAR_NUMERIC:
        cols[c] = frame[c].astype("float64")
    for c in LINEAR_LOG1P:
        x = frame[c].astype("float64")
        assert not (x < 0).any(), f"log1p source {c} has negative values"
        cols[f"log1p_{c}"] = np.log1p(x)
    for c, period in LINEAR_CYCLIC.items():
        ang = 2 * np.pi * frame[c].astype("float64") / period
        cols[f"{c}_sin"], cols[f"{c}_cos"] = np.sin(ang), np.cos(ang)
    return pd.DataFrame(cols, index=frame.index)


def feature_contract_hash(p):
    core = {"tree": [p["tree"]["feature_order"], p["tree"]["dtypes"], p["tree"]["categorical"]],
            "linear": [p["linear"]["feature_order"], p["linear"]["categorical"]]}
    return hashlib.sha256(json.dumps(core, sort_keys=True).encode()).hexdigest()


def fit_encoding_params(frame):
    """Takes the whole frame on purpose: the train filter lives here, and 06.6 proves test rows cannot move it."""
    t = frame.loc[frame["__split"].eq("train")]
    p = {"contract_version": CONTRACT_VERSION, "fit_boundary": "__split == 'train'", "fit_rows": int(len(t))}
    tree_cats = {c: _levels(t[c]) for c in TREE_CATEGORICAL}
    p["tree"] = {"numeric": list(TREE_NUMERIC), "categorical": tree_cats,
                 "feature_order": list(TREE_NUMERIC) + list(TREE_CATEGORICAL),
                 "dtypes": {**{c: "float64" for c in TREE_NUMERIC}, **{c: "category" for c in TREE_CATEGORICAL}},
                 "nan_allowed": list(TREE_NAN_OK)}
    num = linear_numeric_frame(t)
    for c, ind in COVERED_BY_INDICATOR.items():
        assert num[c].isna().eq(t[ind].eq(0)).all(), f"{c}: null mask != ({ind} == 0) on train"
    other_nan = [c for c in num.columns if num[c].isna().any()
                 and c not in COVERED_BY_INDICATOR and c not in EXPLICIT_INDICATORS]
    assert not other_nan, f"undeclared NaN in linear numerics on train: {other_nan}"
    medians = num.median()
    filled = num.fillna(medians)
    for c in EXPLICIT_INDICATORS:
        filled[f"{c}__isna"] = num[c].isna().astype("float64")
    means, stds = filled.mean(), filled.std(ddof=0)
    const = stds.index[stds.eq(0)].tolist()
    assert not const, f"constant linear numerics on train: {const}"
    lin_cats = {}
    for c in LINEAR_CATEGORICAL:
        levels = (["__MISSING__"] if t[c].isna().any() else []) + _levels(t[c])
        lin_cats[c] = [l for l in levels if l not in LINEAR_DROP_LEVELS.get(c, [])]
    numeric_order = list(filled.columns)
    ohe_order = [f"{c}=={l}" for c, levels in lin_cats.items() for l in levels]
    p["linear"] = {"numeric_order": numeric_order, "explicit_indicators": list(EXPLICIT_INDICATORS),
                   "medians": {k: float(v) for k, v in medians.items()},
                   "means": {k: float(v) for k, v in means.items()},
                   "stds": {k: float(v) for k, v in stds.items()},
                   "categorical": lin_cats, "dropped_levels": LINEAR_DROP_LEVELS,
                   "feature_order": numeric_order + ohe_order, "dtype": "float64"}
    p["feature_hash"] = feature_contract_hash(p)
    return p


def encode_tree(frame, p):
    cols = {c: frame[c].astype("float64") for c in p["tree"]["numeric"]}
    for c, levels in p["tree"]["categorical"].items():
        cols[c] = pd.Categorical(frame[c].astype(object).where(frame[c].notna(), None), categories=levels)
    return pd.DataFrame(cols, index=frame.index)[p["tree"]["feature_order"]]


def encode_linear(frame, p):
    lp = p["linear"]
    num = linear_numeric_frame(frame)
    for c in lp["explicit_indicators"]:
        num[f"{c}__isna"] = num[c].isna().astype("float64")
    num = num.fillna(pd.Series(lp["medians"]))
    num = (num[lp["numeric_order"]] - pd.Series(lp["means"])) / pd.Series(lp["stds"])
    cols = {}
    for c, levels in lp["categorical"].items():
        s = frame[c].astype(object).where(frame[c].notna(), "__MISSING__").astype(str)
        for l in levels:
            cols[f"{c}=={l}"] = s.eq(l).astype("float64")
    out = pd.concat([num, pd.DataFrame(cols, index=frame.index)], axis=1)[lp["feature_order"]].astype("float64")
    assert np.isfinite(out.to_numpy()).all(), "non-finite values in linear encoding"
    return out


P = fit_encoding_params(df)
print("fit rows:", P["fit_rows"], "| tree features:", len(P["tree"]["feature_order"]),
      "| linear features:", len(P["linear"]["feature_order"]), "| feature hash:", P["feature_hash"][:16])

In [ ]:
# 06.6 Leakage self-checks
_dump = lambda p: json.dumps(p, sort_keys=True)
# (a) the fit ignores test rows: train-only input and train + corrupted test copies give identical parameters
assert _dump(fit_encoding_params(df.loc[tr])) == _dump(P), "encoding params depend on non-train rows"
junk = df.loc[~tr].sample(n=min(20_000, int((~tr).sum())), random_state=SEED).copy()
for c in junk.columns:
    if c == "__split":
        continue
    if pd.api.types.is_numeric_dtype(junk[c]) and not pd.api.types.is_bool_dtype(junk[c]):
        junk[c] = junk[c] * 7 + 1000
    elif junk[c].dtype == object:
        junk[c] = "CORRUPT"
assert _dump(fit_encoding_params(pd.concat([df.loc[tr], junk], ignore_index=True))) == _dump(P), \
    "encoding params moved when corrupted test rows were appended"

# (b) contemporaneous: corrupting label / incumbent columns must leave every encoded feature bit-identical
sample = df.sample(n=min(20_000, len(df)), random_state=SEED)
bent = sample.copy()
rng = np.random.default_rng(SEED)
bent["is_fraud"] = 1 - bent["is_fraud"]
bent["sample_weight"] = rng.uniform(0, 1, len(bent))
bent["label_source"] = rng.permutation(bent["label_source"].to_numpy())
bent["payment_risk_score"] = "0"
bent["alerted_flag"] = np.where(bent["alerted_flag"].eq("Y"), "N", "Y")
bent["baseline_score"] = 0.0
bent["is_retry"] = 1 - bent["is_retry"]
pd.testing.assert_frame_equal(encode_tree(sample, P), encode_tree(bent, P))
pd.testing.assert_frame_equal(encode_linear(sample, P), encode_linear(bent, P))
LEAKAGE_CHECKS = {"fit_ignores_test_rows": True, "fit_ignores_corrupted_test_rows": True,
                  "features_independent_of_label_and_incumbent_columns": True}
print("leakage self-checks passed:", LEAKAGE_CHECKS)

In [ ]:
# 06.7 Encode, verify NaN policy and unseen levels, VIF (information only), save
Xt = encode_tree(df, P)
Xl = encode_linear(df, P)
assert list(Xt.columns) == P["tree"]["feature_order"] and list(Xl.columns) == P["linear"]["feature_order"]

nan_train = Xt.loc[tr].isna().sum()
undeclared = [c for c in nan_train.index[nan_train > 0] if c not in P["tree"]["nan_allowed"]]
assert not undeclared, f"undeclared NaN in tree features on train: {undeclared}"
unseen = {c: int((df[c].notna() & Xt[c].isna()).sum()) for c in TREE_CATEGORICAL}
unseen_train = {c: int((df.loc[tr, c].notna() & Xt.loc[tr, c].isna()).sum()) for c in TREE_CATEGORICAL}
assert not any(unseen_train.values()), unseen_train
print("unseen categorical values (test rows -> NaN/all-zero):", {k: v for k, v in unseen.items() if v} or "none")

Z = Xl.loc[tr, P["linear"]["numeric_order"]].to_numpy()
R = np.corrcoef(Z, rowvar=False)
vif = pd.Series(np.diag(np.linalg.pinv(R)), index=P["linear"]["numeric_order"]).sort_values(ascending=False)
print("\nVIF on standardised linear numerics (train) — information only, never a selector:")
print(vif.head(15).round(2).to_string())

tree_out = pd.concat([df[META], Xt], axis=1)
lin_out = pd.concat([df[META], Xl], axis=1)
assert tree_out.columns.is_unique and lin_out.columns.is_unique
names = pd.DataFrame(
    [{"family": "tree", "position": i, "feature": c, "dtype": P["tree"]["dtypes"][c],
      "kind": "categorical" if c in TREE_CATEGORICAL else "numeric"} for i, c in enumerate(P["tree"]["feature_order"])]
    + [{"family": "linear", "position": i, "feature": c, "dtype": "float64",
        "kind": "one-hot" if "==" in c else ("indicator" if c.endswith("__isna") else "numeric")}
       for i, c in enumerate(P["linear"]["feature_order"])])
save_s3(tree_out, "data/06_encoded_tree.parquet")
save_s3(lin_out, "data/06_encoded_linear.parquet")
save_s3(names, "data/06_feature_names.parquet")
save_json_s3(P, "data/06_encoding_params.json")

record = {
    "notebook": "06_Correlation_VIF_Encoding", "marker": MARKER,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "rows": int(len(df)), "fit_boundary": "__split == 'train'", "fit_rows": P["fit_rows"],
    "acknowledged_sentinels": ACKNOWLEDGED_SENTINELS,
    "identities_asserted": IDENTITIES, "excluded": EXCLUDED, "meta_columns": META,
    "tree_features": len(P["tree"]["feature_order"]), "linear_features": len(P["linear"]["feature_order"]),
    "feature_hash": P["feature_hash"],
    "high_corr_pairs": high_corr.to_dict("records"), "vif_top": vif.head(15).round(3).to_dict(),
    "unseen_levels_all_rows": unseen, "leakage_checks": LEAKAGE_CHECKS,
    "selection_rule": "by name only; correlation and VIF are informational",
    "library_versions": LIB_VERSIONS, "version_drift": VERSION_DRIFT,
}
save_json_s3(record, "reports/06_run_record.json")